# Day 2: Comparing 3D cell-center detectors and improving tracking

This notebook improves the Day 1 baseline in two controlled steps:

1. compare several 3D center-detection transformations on annotated training frames;
2. compare linking methods on complete training movies using the published edge metric.

The selected detector and tracker are then used to create `submission.csv`. All selection uses training data. The test images are used only for final inference.

## Why these methods

Comparative fluorescence-microscopy studies found that supervised methods are strongest at very low signal-to-noise ratios, while h-dome and multiscale methods are competitive unsupervised choices. The Cell Tracking Challenge reports that detection-driven learned methods are especially effective for clustered and embryonic data. Multiscale Laplacian-of-Gaussian center detection has also been used successfully on 3D challenge datasets.

This notebook tests an interpretable classical sequence before moving to a learned model:

```text
robust normalization
        ↓
local-background removal
        ↓
anisotropic multiscale LoG
        ↓
local peak selection in physical space
        ↓
Hungarian or motion-aware Hungarian linking
```

References:

- [Smal et al., Quantitative comparison of spot detection methods](https://pubmed.ncbi.nlm.nih.gov/19556194/)
- [Štěpka et al., Performance and sensitivity of 3D spot detection](https://onlinelibrary.wiley.com/doi/full/10.1002/cyto.a.22692)
- [Maška et al., The Cell Tracking Challenge: 10 years of benchmarking](https://pmc.ncbi.nlm.nih.gov/articles/PMC10333123/)
- [Eschweiler and Stegmaier, multiscale LoG detection in 3D](https://arxiv.org/abs/1904.06890)
- [Faure et al., 3D+time analysis of developing organisms](https://www.nature.com/articles/ncomms9674)
- [Sugawara et al., ELEPHANT sparse-annotation detection and flow](https://elifesciences.org/articles/69380)

Implementations below are original to this notebook and use these papers for methodological guidance.

In [ ]:
from pathlib import Path
from itertools import product
from collections import Counter, defaultdict
import csv, json, math, time, warnings

import blosc2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter, gaussian_laplace, maximum_filter, minimum_filter
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree

warnings.filterwarnings('ignore')
np.random.seed(42)
print('numpy', np.__version__, '| scipy available | blosc2', blosc2.__version__)

## 1. Configuration

Spatial settings are expressed in microns and converted to voxel units. This is essential because the axial resolution is four times coarser than the lateral resolution.

In [ ]:
COMPETITION_SLUG = 'biohub-cell-tracking-during-development'
SPACING_UM = np.array([1.625, 0.40625, 0.40625], dtype=np.float32)
MATCH_RADIUS_UM = 7.0

SCREEN_METHODS = ['dogSingle', 'logSingle', 'logMulti', 'backgroundLogMulti', 'prominentLogMulti']
SCREEN_QUANTILES = [0.992, 0.995, 0.997]
SCREEN_NMS_UM = [2.0, 2.5]
SCREEN_FRAMES_PER_MOVIE = 3
SCREEN_MOVIES_PER_EMBRYO = 2
VALIDATION_MOVIES_PER_EMBRYO = 1

BASE_CONFIG = {
    'small_sigma_um': 1.0,
    'large_sigma_um': 2.2,
    'log_scales_um': [1.0, 1.5],
    'background_sigma_um': 4.0,
    'prominence_radius_um': 4.0,
    'max_nodes_per_frame': 1800,
    'max_link_um': 7.0,
    'motion_gate_um': 10.0,
    'motion_weight': 0.65,
}
print(json.dumps(BASE_CONFIG, indent=2))

## 2. Dependency-free Zarr v3 readers

The competition images and GEFF annotations use Zarr v3. These compact readers load the numeric arrays needed here using Kaggle's existing `blosc2` package, so the committed notebook remains offline.

In [ ]:
def _dtype_from_metadata(meta):
    dtype = np.dtype(meta['data_type'])
    byte_codec = next((c for c in meta.get('codecs', []) if c.get('name') == 'bytes'), None)
    if byte_codec and dtype.itemsize > 1:
        endian = byte_codec.get('configuration', {}).get('endian', 'little')
        dtype = dtype.newbyteorder('<' if endian == 'little' else '>')
    return dtype

def read_zarr_v3_array(array_path):
    array_path = Path(array_path)
    meta = json.loads((array_path / 'zarr.json').read_text())
    shape = tuple(meta['shape'])
    chunks = tuple(meta['chunk_grid']['configuration']['chunk_shape'])
    dtype = _dtype_from_metadata(meta)
    fill = meta.get('fill_value', 0)
    out = np.full(shape, 0 if fill is None else fill, dtype=dtype)
    grid = tuple(math.ceil(s / c) for s, c in zip(shape, chunks))
    for index in product(*(range(n) for n in grid)):
        path = array_path / 'c'
        for i in index:
            path /= str(i)
        if not path.exists():
            continue
        target = tuple(slice(i*c, min((i+1)*c, s)) for i, c, s in zip(index, chunks, shape))
        actual_shape = tuple(sl.stop-sl.start for sl in target)
        flat = np.frombuffer(blosc2.decompress(path.read_bytes()), dtype=dtype)
        if flat.size == math.prod(chunks):
            chunk = flat.reshape(chunks)[tuple(slice(0,n) for n in actual_shape)]
        elif flat.size == math.prod(actual_shape):
            chunk = flat.reshape(actual_shape)
        else:
            raise ValueError(f'Unexpected decoded chunk size at {path}: {flat.size}')
        out[target] = chunk
    return out

class TimeChunkedZarr:
    def __init__(self, path):
        self.path = Path(path) / '0'
        self.metadata = json.loads((self.path / 'zarr.json').read_text())
        self.shape = tuple(self.metadata['shape'])
        self.chunks = tuple(self.metadata['chunk_grid']['configuration']['chunk_shape'])
        self.dtype = _dtype_from_metadata(self.metadata)
        if self.chunks != (1,) + self.shape[1:]:
            raise ValueError(f'Expected one full frame per chunk, found {self.chunks}')

    def __len__(self):
        return self.shape[0]

    def __getitem__(self, t):
        t = int(t) % self.shape[0]
        path = self.path / 'c' / str(t) / '0' / '0' / '0'
        raw = blosc2.decompress(path.read_bytes())
        return np.frombuffer(raw, dtype=self.dtype).reshape(self.chunks)[0]

def find_competition_root():
    candidates = [Path('/kaggle/input/competitions') / COMPETITION_SLUG, Path('/kaggle/input') / COMPETITION_SLUG]
    for path in candidates:
        if (path / 'train').exists() and (path / 'test').exists():
            return path
    raise FileNotFoundError('Attach the official competition data.')

ROOT = find_competition_root()
TRAIN_DIR, TEST_DIR = ROOT / 'train', ROOT / 'test'
train_movies = sorted(TRAIN_DIR.glob('*.zarr'))
test_movies = sorted(TEST_DIR.glob('*.zarr'))
print(ROOT, '| train:', len(train_movies), '| test:', len(test_movies))

In [ ]:
def _first_existing(base, relative_paths):
    for rel in relative_paths:
        path = base / rel
        if (path / 'zarr.json').exists():
            return path
    raise FileNotFoundError(f'None of {relative_paths} found under {base}')

def _recursive_find_key(obj, target):
    if isinstance(obj, dict):
        if target in obj:
            return obj[target]
        for value in obj.values():
            found = _recursive_find_key(value, target)
            if found is not None:
                return found
    elif isinstance(obj, list):
        for value in obj:
            found = _recursive_find_key(value, target)
            if found is not None:
                return found
    return None

def load_geff(geff_path):
    geff_path = Path(geff_path)
    ids = read_zarr_v3_array(_first_existing(geff_path, ['nodes/ids'])).reshape(-1).astype(np.int64)
    props = {}
    for key in ['t','z','y','x']:
        props[key] = read_zarr_v3_array(_first_existing(geff_path, [f'nodes/props/{key}/values', f'nodes/{key}'])).reshape(-1)
    edge_path = _first_existing(geff_path, ['edges/ids'])
    edges = read_zarr_v3_array(edge_path).reshape(-1, 2).astype(np.int64)
    nodes = pd.DataFrame({'node_id': ids, **props})
    root_meta = json.loads((geff_path / 'zarr.json').read_text())
    estimated = _recursive_find_key(root_meta, 'estimated_number_of_nodes')
    return nodes, edges, float(estimated) if estimated is not None else np.nan

# Verify the reader on one annotation graph before running experiments.
probe = train_movies[0]
probe_nodes, probe_edges, probe_estimated = load_geff(probe.with_suffix('.geff'))
print(probe.stem, '| GT nodes:', len(probe_nodes), '| GT edges:', len(probe_edges), '| estimated total:', probe_estimated)
display(probe_nodes.head())

## 3. Candidate detection methods

Each method returns a response volume in which larger values indicate stronger center evidence. LoG filters are scale-normalized and anisotropic: a scale in microns is converted separately for `Z`, `Y`, and `X`.

In [ ]:
def sigma_vox(sigma_um):
    return tuple((float(sigma_um) / SPACING_UM).tolist())

def odd_window(radius_um):
    radius = np.maximum(1, np.ceil(float(radius_um) / SPACING_UM).astype(int))
    return tuple((2 * radius + 1).tolist())

def normalize_frame(frame):
    frame = np.asarray(frame, dtype=np.float32)
    sample = frame[::max(1, frame.shape[0]//32), ::4, ::4]
    lo, hi = np.quantile(sample, [0.01, 0.9995])
    return np.clip((frame - lo) / max(float(hi-lo), 1.0), 0, 1)

def response_volume(image, method, config=BASE_CONFIG):
    if method == 'dogSingle':
        return gaussian_filter(image, sigma_vox(config['small_sigma_um'])) - gaussian_filter(image, sigma_vox(config['large_sigma_um']))

    source = image
    if method in {'backgroundLogMulti', 'prominentLogMulti'}:
        background = gaussian_filter(image, sigma_vox(config['background_sigma_um']))
        source = np.clip(image - background, 0, None)

    scales = [config['small_sigma_um']] if method == 'logSingle' else config['log_scales_um']
    responses = [-float(scale)**2 * gaussian_laplace(source, sigma_vox(scale)) for scale in scales]
    response = np.maximum.reduce(responses)

    if method == 'prominentLogMulti':
        window = odd_window(config['prominence_radius_um'])
        local_floor = maximum_filter(minimum_filter(response, size=window), size=window)
        response = np.maximum(response - local_floor, 0)
    return response.astype(np.float32, copy=False)

def peaks_from_response(response, quantile, nms_um, max_nodes):
    positive = response[response > 0]
    if positive.size == 0:
        return np.empty((0,3), np.int32), np.empty(0, np.float32)
    threshold = np.quantile(positive, quantile)
    maxima = response == maximum_filter(response, size=odd_window(nms_um), mode='nearest')
    coords = np.argwhere(maxima & (response >= threshold))
    scores = response[tuple(coords.T)] if len(coords) else np.empty(0, np.float32)
    if len(coords) > max_nodes:
        keep = np.argpartition(scores, -max_nodes)[-max_nodes:]
        coords, scores = coords[keep], scores[keep]
    order = np.argsort(scores)[::-1]
    return coords[order].astype(np.int32), scores[order].astype(np.float32)

def detect_frame(frame, detector):
    image = normalize_frame(frame)
    response = response_volume(image, detector['method'])
    return peaks_from_response(response, detector['quantile'], detector['nms_um'], BASE_CONFIG['max_nodes_per_frame'])

## 4. Sparse-label-aware screening

Unmatched predictions are not automatically false positives because the annotations are sparse. The screening score therefore combines three measurable quantities:

- recall of annotated centers within 7 µm;
- duplicate predictions around the same annotation;
- agreement with the provided estimated total node count, when available.

Response volumes are computed once per frame and reused for every threshold and NMS setting.

In [ ]:
def embryo_id(path):
    return Path(path).stem.split('_')[0]

def balanced_movies(paths, per_embryo):
    groups = defaultdict(list)
    for path in paths:
        groups[embryo_id(path)].append(path)
    return [p for key in sorted(groups) for p in groups[key][:per_embryo]]

def representative_times(nodes, n):
    counts = nodes.groupby('t').size().sort_index()
    available = counts.index.to_numpy(dtype=int)
    if len(available) <= n:
        return available.tolist()
    targets = np.linspace(available.min(), available.max(), n)
    return sorted(set(int(available[np.argmin(abs(available-x))]) for x in targets))

def match_centers(pred_coords, gt_coords, radius_um=MATCH_RADIUS_UM):
    if len(pred_coords) == 0 or len(gt_coords) == 0:
        return [], len(gt_coords), 0
    distances = np.linalg.norm((pred_coords[:,None,:] - gt_coords[None,:,:]) * SPACING_UM, axis=2)
    rows, cols = linear_sum_assignment(distances)
    pairs = [(int(r), int(c), float(distances[r,c])) for r,c in zip(rows,cols) if distances[r,c] <= radius_um]
    near_counts = (distances <= radius_um).sum(axis=0)
    duplicates = int(np.maximum(near_counts - 1, 0).sum())
    return pairs, len(gt_coords)-len(pairs), duplicates

screen_movies = balanced_movies(train_movies, SCREEN_MOVIES_PER_EMBRYO)
screen_rows = []
for movie_path in screen_movies:
    print('Screening', movie_path.stem)
    movie = TimeChunkedZarr(movie_path)
    gt_nodes, _, estimated = load_geff(movie_path.with_suffix('.geff'))
    expected_per_frame = estimated / len(movie) if np.isfinite(estimated) else np.nan
    for t in representative_times(gt_nodes, SCREEN_FRAMES_PER_MOVIE):
        image = normalize_frame(movie[t])
        gt = gt_nodes.loc[gt_nodes.t == t, ['z','y','x']].to_numpy(dtype=float)
        for method in SCREEN_METHODS:
            response = response_volume(image, method)
            for quantile in SCREEN_QUANTILES:
                for nms_um in SCREEN_NMS_UM:
                    coords, _ = peaks_from_response(response, quantile, nms_um, BASE_CONFIG['max_nodes_per_frame'])
                    pairs, misses, duplicates = match_centers(coords, gt)
                    recall = len(pairs) / len(gt) if len(gt) else np.nan
                    count_ratio = len(coords) / expected_per_frame if expected_per_frame and np.isfinite(expected_per_frame) else np.nan
                    screen_rows.append({'dataset':movie_path.stem,'t':t,'method':method,'quantile':quantile,'nms_um':nms_um,
                                        'gt_nodes':len(gt),'pred_nodes':len(coords),'matched':len(pairs),'misses':misses,
                                        'duplicates':duplicates,'recall7':recall,'count_ratio':count_ratio})

screen = pd.DataFrame(screen_rows)
screen.to_csv('/kaggle/working/detectorScreen.csv', index=False)
display(screen.head())

In [ ]:
summary = (screen.groupby(['method','quantile','nms_um'], as_index=False)
           .agg(matched=('matched','sum'), gt_nodes=('gt_nodes','sum'), duplicates=('duplicates','sum'),
                mean_count_ratio=('count_ratio','mean'), frames=('t','count')))
summary['recall7'] = summary.matched / summary.gt_nodes
summary['duplicate_rate'] = summary.duplicates / summary.gt_nodes.clip(lower=1)
ratio_penalty = np.where(summary.mean_count_ratio.notna(), np.abs(np.log(summary.mean_count_ratio.clip(lower=.05))), 0)
summary['screen_score'] = summary.recall7 - 0.05*summary.duplicate_rate - 0.10*ratio_penalty
summary = summary.sort_values(['screen_score','recall7'], ascending=False).reset_index(drop=True)
summary.to_csv('/kaggle/working/detectorScreenSummary.csv', index=False)
display(summary.head(12))
best_row = summary.iloc[0]
BEST_DETECTOR = {'method':best_row.method, 'quantile':float(best_row['quantile']), 'nms_um':float(best_row.nms_um)}
print('Selected detector for full-movie validation:', BEST_DETECTOR)

## 5. Linking methods

The Day 1 greedy linker takes the shortest available edge first. Hungarian assignment instead chooses the lowest-cost set of links for the whole frame pair. The motion-aware version predicts each cell's next position from its previous displacement and combines predicted-position distance with ordinary displacement.

In [ ]:
def greedy_links(prev_coords, prev_ids, curr_coords, curr_ids, max_um):
    if not len(prev_coords) or not len(curr_coords): return []
    a, b = prev_coords*SPACING_UM, curr_coords*SPACING_UM
    tree, candidates = cKDTree(b), []
    for i, js in enumerate(tree.query_ball_point(a, r=max_um)):
        for j in js: candidates.append((float(np.linalg.norm(a[i]-b[j])), i, j))
    used_a, used_b, links = set(), set(), []
    for distance, i, j in sorted(candidates):
        if i not in used_a and j not in used_b:
            links.append((int(prev_ids[i]), int(curr_ids[j]), distance))
            used_a.add(i); used_b.add(j)
    return links

def assignment_links(prev_coords, prev_ids, curr_coords, curr_ids, velocities, method):
    if not len(prev_coords) or not len(curr_coords): return []
    prev_um, curr_um = prev_coords*SPACING_UM, curr_coords*SPACING_UM
    raw = np.linalg.norm(prev_um[:,None,:]-curr_um[None,:,:], axis=2)
    if method == 'hungarian':
        cost = raw.copy()
        valid = raw <= BASE_CONFIG['max_link_um']
    else:
        velocity = np.array([velocities.get(int(i), np.zeros(3)) for i in prev_ids])
        predicted = prev_um + velocity
        motion = np.linalg.norm(predicted[:,None,:]-curr_um[None,:,:], axis=2)
        w = BASE_CONFIG['motion_weight']
        cost = w*motion + (1-w)*raw
        valid = (raw <= BASE_CONFIG['max_link_um']) & (motion <= BASE_CONFIG['motion_gate_um'])
    large = 1e6
    rows, cols = linear_sum_assignment(np.where(valid, cost, large))
    return [(int(prev_ids[i]), int(curr_ids[j]), float(raw[i,j])) for i,j in zip(rows,cols) if valid[i,j]]

def process_movie(path, detector, linker='motionHungarian', verbose=False):
    movie = TimeChunkedZarr(path)
    nodes, edges, next_id = [], [], 1
    prev_coords = np.empty((0,3), np.int32); prev_ids = np.empty(0, np.int64)
    velocities = {}
    started = time.time()
    for t in range(len(movie)):
        coords, scores = detect_frame(movie[t], detector)
        ids = np.arange(next_id, next_id+len(coords), dtype=np.int64); next_id += len(coords)
        nodes.extend((int(i),t,int(z),int(y),int(x),float(s)) for i,(z,y,x),s in zip(ids,coords,scores))
        if linker == 'greedy':
            links = greedy_links(prev_coords, prev_ids, coords, ids, BASE_CONFIG['max_link_um'])
        else:
            links = assignment_links(prev_coords, prev_ids, coords, ids, velocities, linker)
        prev_lookup = {int(i): c*SPACING_UM for i,c in zip(prev_ids,prev_coords)}
        curr_lookup = {int(i): c*SPACING_UM for i,c in zip(ids,coords)}
        new_velocities = {}
        for source,target,distance in links:
            edges.append((source,target,distance))
            observed = curr_lookup[target]-prev_lookup[source]
            old = velocities.get(source, observed)
            new_velocities[target] = 0.5*old + 0.5*observed
        velocities = new_velocities
        prev_coords, prev_ids = coords, ids
        if verbose and (t == 0 or (t+1)%10 == 0 or t+1 == len(movie)):
            print(f'  {Path(path).stem}: {t+1:>3}/{len(movie)} frames | {len(nodes):,} nodes | {len(edges):,} edges')
    return nodes, edges, (time.time()-started)

## 6. Complete-movie edge validation

The local scorer follows the published edge definition: predicted and annotated nodes are optimally matched within 7 µm at each timepoint; predicted edges become true positives when their matched endpoints form an annotated edge. The exact official package remains the final authority. Division scoring is not used to select this division-free baseline.

In [ ]:
def score_edges(pred_nodes, pred_edges, gt_nodes, gt_edges, estimated_total):
    pred_df = pd.DataFrame(pred_nodes, columns=['node_id','t','z','y','x','score'])
    mapping = {}
    for t, gt_part in gt_nodes.groupby('t'):
        pred_part = pred_df[pred_df.t == t]
        if pred_part.empty or gt_part.empty: continue
        pred_xyz = pred_part[['z','y','x']].to_numpy(float)
        gt_xyz = gt_part[['z','y','x']].to_numpy(float)
        distances = np.linalg.norm((pred_xyz[:,None,:]-gt_xyz[None,:,:])*SPACING_UM, axis=2)
        rows, cols = linear_sum_assignment(distances)
        pred_ids = pred_part.node_id.to_numpy(); gt_ids = gt_part.node_id.to_numpy()
        for i,j in zip(rows,cols):
            if distances[i,j] <= MATCH_RADIUS_UM: mapping[int(pred_ids[i])] = int(gt_ids[j])

    gt_edge_set = {tuple(map(int,e)) for e in gt_edges}
    gt_out = {s for s,_ in gt_edge_set}; gt_in = {t for _,t in gt_edge_set}
    tp_pairs, fp = set(), 0
    for source,target,_ in pred_edges:
        ms, mt = mapping.get(source), mapping.get(target)
        pair = (ms,mt)
        if ms is not None and mt is not None and pair in gt_edge_set:
            tp_pairs.add(pair)
        elif (ms is not None and ms in gt_out) or (mt is not None and mt in gt_in):
            fp += 1
    tp = len(tp_pairs); fn = len(gt_edge_set - tp_pairs)
    jaccard = tp / max(tp+fp+fn, 1)
    ratio = (len(pred_nodes)-estimated_total)/estimated_total if np.isfinite(estimated_total) and estimated_total>0 else np.nan
    adjusted = max(0, jaccard*(1-0.1*ratio)) if np.isfinite(ratio) else jaccard
    return {'edge_tp':tp,'edge_fp':fp,'edge_fn':fn,'edge_jaccard':jaccard,'node_count':len(pred_nodes),
            'node_ratio_delta':ratio,'adjusted_edge_jaccard':adjusted,'matched_nodes':len(set(mapping.values()))}

DAY1_DETECTOR = {'method':'dogSingle','quantile':0.995,'nms_um':2.5}
validation_movies = balanced_movies(train_movies, VALIDATION_MOVIES_PER_EMBRYO)
validation_configs = [
    ('day1Greedy', DAY1_DETECTOR, 'greedy'),
    ('bestGreedy', BEST_DETECTOR, 'greedy'),
    ('bestHungarian', BEST_DETECTOR, 'hungarian'),
    ('bestMotionHungarian', BEST_DETECTOR, 'motionHungarian'),
]
validation_rows = []
for movie_path in validation_movies:
    gt_nodes, gt_edges, estimated = load_geff(movie_path.with_suffix('.geff'))
    for label, detector, linker in validation_configs:
        print(movie_path.stem, label)
        nodes, edges, seconds = process_movie(movie_path, detector, linker)
        metrics = score_edges(nodes, edges, gt_nodes, gt_edges, estimated)
        validation_rows.append({'dataset':movie_path.stem,'configuration':label,'linker':linker,
                                'detector':json.dumps(detector,sort_keys=True),'seconds':seconds,
                                'pred_edges':len(edges), **metrics})

validation = pd.DataFrame(validation_rows)
validation.to_csv('/kaggle/working/fullMovieValidation.csv', index=False)
display(validation)

In [ ]:
validation['edge_weight'] = validation.edge_tp + validation.edge_fp + validation.edge_fn
summary_rows = []
for name, group in validation.groupby('configuration'):
    tp, fp, fn = group[['edge_tp','edge_fp','edge_fn']].sum()
    weights = group.edge_weight.to_numpy()
    adjusted = np.average(group.adjusted_edge_jaccard, weights=weights) if weights.sum() else np.nan
    summary_rows.append({'configuration':name,'adjusted_edge_jaccard':adjusted,
                         'edge_jaccard':tp/max(tp+fp+fn,1),'matched_nodes':group.matched_nodes.sum(),
                         'edge_tp':tp,'edge_fp':fp,'edge_fn':fn,'seconds':group.seconds.sum()})
config_summary = pd.DataFrame(summary_rows)
config_summary = config_summary.sort_values(['adjusted_edge_jaccard','edge_jaccard'], ascending=False).reset_index(drop=True)
display(config_summary)
selected_label = config_summary.iloc[0].configuration
selected = next(x for x in validation_configs if x[0] == selected_label)
FINAL_DETECTOR, FINAL_LINKER = selected[1], selected[2]
print('Selected for submission:', selected_label, FINAL_DETECTOR, FINAL_LINKER)

## 7. Visual check

The overlay below is a sanity check, not the selection criterion. Annotated centers are yellow and predictions are cyan. Because annotations are sparse, a cyan-only point may still be a real cell.

In [ ]:
viz_path = validation_movies[0]
viz_movie = TimeChunkedZarr(viz_path)
viz_gt, _, _ = load_geff(viz_path.with_suffix('.geff'))
viz_t = representative_times(viz_gt, 3)[1]
viz_frame = np.asarray(viz_movie[viz_t], dtype=np.float32)
viz_pred, _ = detect_frame(viz_frame, FINAL_DETECTOR)
viz_truth = viz_gt.loc[viz_gt.t == viz_t, ['z','y','x']].to_numpy()
sample = viz_frame[::2,::4,::4]; vmin,vmax=np.quantile(sample,[.5,.999])
fig,ax=plt.subplots(figsize=(8,8)); ax.imshow(viz_frame.max(axis=0),cmap='gray',vmin=vmin,vmax=vmax)
if len(viz_pred): ax.scatter(viz_pred[:,2],viz_pred[:,1],s=12,facecolors='none',edgecolors='cyan',label='predicted')
if len(viz_truth): ax.scatter(viz_truth[:,2],viz_truth[:,1],s=32,facecolors='none',edgecolors='yellow',label='annotated')
ax.legend(); ax.set_title(f'{viz_path.stem}, t={viz_t}'); ax.axis('off'); plt.show()

## 8. Test inference and submission

The chosen configuration is now frozen. Every test movie is processed once, and its node, edge, and runtime counts are recorded before the final CSV is assembled.

In [ ]:
COLUMNS = ['dataset','row_type','node_id','t','z','y','x','source_id','target_id']
all_parts, run_rows = [], []
for movie_path in test_movies:
    print('Processing', movie_path.stem)
    nodes, edges, seconds = process_movie(movie_path, FINAL_DETECTOR, FINAL_LINKER, verbose=True)
    node_rows = [(movie_path.stem,'node',i,t,z,y,x,-1,-1) for i,t,z,y,x,score in nodes]
    edge_rows = [(movie_path.stem,'edge',-1,-1,-1,-1,-1,s,t) for s,t,d in edges]
    part = pd.DataFrame(node_rows+edge_rows, columns=COLUMNS)
    all_parts.append(part)
    run_rows.append({'dataset':movie_path.stem,'nodes':len(nodes),'edges':len(edges),'seconds':seconds})

submission = pd.concat(all_parts, ignore_index=True)
submission.insert(0, 'id', np.arange(len(submission), dtype=np.int64))
for col in ['id','node_id','t','z','y','x','source_id','target_id']:
    submission[col] = submission[col].astype(np.int64)
run_summary = pd.DataFrame(run_rows)
display(run_summary)

In [ ]:
def validate_submission(df, expected_datasets):
    assert list(df.columns) == ['id'] + COLUMNS
    assert not df.empty and df.id.is_unique
    assert set(df.dataset) == set(expected_datasets)
    assert set(df.row_type) <= {'node','edge'}
    for dataset, part in df.groupby('dataset'):
        nodes = part[part.row_type == 'node']; edges = part[part.row_type == 'edge']
        assert len(nodes) and nodes.node_id.is_unique
        times = dict(zip(nodes.node_id, nodes.t)); ids = set(times)
        assert set(edges.source_id) <= ids and set(edges.target_id) <= ids
        if len(edges):
            assert edges.source_id.value_counts().max() <= 1
            assert edges.target_id.value_counts().max() <= 1
            assert all(times[b] == times[a]+1 for a,b in zip(edges.source_id,edges.target_id))
    return True

assert validate_submission(submission, [p.stem for p in test_movies])
submission.to_csv('/kaggle/working/submission.csv', index=False)
run_summary.to_csv('/kaggle/working/runSummary.csv', index=False)
with open('/kaggle/working/selectedConfiguration.json','w') as f:
    json.dump({'detector':FINAL_DETECTOR,'linker':FINAL_LINKER,'baseConfig':BASE_CONFIG},f,indent=2)
print(f'Wrote {len(submission):,} rows to /kaggle/working/submission.csv')
print('Final configuration:', FINAL_DETECTOR, FINAL_LINKER)

## Reading the result

The notebook saves four diagnostic files in addition to the required submission:

- `detectorScreen.csv`: frame-level detector results;
- `detectorScreenSummary.csv`: aggregated detector ranking;
- `fullMovieValidation.csv`: graph-level comparison on complete training movies;
- `selectedConfiguration.json`: exact detector and tracker used for the submission.

A higher public score is useful evidence, but the full-movie validation table is the main record of why this configuration was selected.